# Hyperparameter Tuning — GAT & GATv2 on QM9

This notebook runs **random search** hyperparameter tuning for both GAT (GATConv) and GATv2 (GATv2Conv) on the QM9 molecular property regression task.

---

## Why random search over grid search?

A full grid search over 5 hyperparameters with 3 values each gives 243 combinations per model — far too many for Colab. Random search samples configurations uniformly from the same space and in practice finds near-optimal configurations in 20–30 trials, because most hyperparameter landscapes have a small number of dimensions that actually matter. Any trial that gets a bad `lr` is wasted regardless of the other settings — random search wastes fewer trials on those dead zones than a grid would.

## Search space

| Hyperparameter | Values | Notes |
|---|---|---|
| `hidden_dim` | 64, 128, 256 | Must be divisible by `heads` |
| `num_layers` | 2, 3, 4 | Message passing depth |
| `heads` | 2, 4, 8 | Attention heads |
| `dropout` | 0.0, 0.1, 0.2 | Applied to coefficients and node repr |
| `lr` | 1e-3, 5e-4, 1e-4 | Adam learning rate |
| `share_weights` | True, False | GATv2 only — whether W_l == W_r |

Combinations where `hidden_dim % heads != 0` are automatically skipped.

## What this notebook produces

For each trial:
- Per-epoch validation MAE and MSE logged to CSV
- Training curve saved as PNG
- Best checkpoint saved to Drive

After all trials:
- Hyperparameter sensitivity plots (one variable at a time)
- Pairwise heatmap for `hidden_dim` × `num_layers`
- GAT vs GATv2 head-to-head scatter
- Parameter count vs val MAE
- Best config per model evaluated on test set
- Final comparison training curves (best GAT vs best GATv2)

---

## Steps

1. **Cell 1** — Mount Google Drive
2. **Cell 2** — Install dependencies
3. **Cell 3** — Clone project repo
4. **Cell 4** — Setup: device, paths, shared data loaders
5. **Cell 5** — Define search space and random sampler
6. **Cell 6** — Run random search for GAT
7. **Cell 7** — Run random search for GATv2
8. **Cell 8** — Hyperparameter sensitivity plots
9. **Cell 9** — Pairwise heatmap: hidden_dim × num_layers
10. **Cell 10** — GAT vs GATv2 head-to-head scatter
11. **Cell 11** — Parameter count vs val MAE
12. **Cell 12** — Test set evaluation for best configs
13. **Cell 13** — Final comparison training curves

In [1]:
# ── Cell 1: Mount Google Drive ───────────────────────────────────────────────
#
# All checkpoints, logs, and plots are written to Google Drive so they
# persist across Colab sessions. The tuning run may take several hours —
# saving to Drive means a session timeout doesn't lose your results.
#
# Expected Drive structure after this cell:
#   gnn_qm9/
#     data/
#     outputs/
#       checkpoints/tuning/gat/
#       checkpoints/tuning/gatv2/
#       logs/tuning/gat/
#       logs/tuning/gatv2/
#       plots/tuning/
#       results/

from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/gnn_qm9_final_full'

for path in [
    f'{DRIVE_ROOT}/data',
    f'{DRIVE_ROOT}/outputs/checkpoints/tuning/gat',
    f'{DRIVE_ROOT}/outputs/checkpoints/tuning/gatv2',
    f'{DRIVE_ROOT}/outputs/logs/tuning/gat',
    f'{DRIVE_ROOT}/outputs/logs/tuning/gatv2',
    f'{DRIVE_ROOT}/outputs/plots/tuning',
    f'{DRIVE_ROOT}/outputs/results',
]:
    os.makedirs(path, exist_ok=True)

print('Drive mounted.')
print('Output root:', DRIVE_ROOT)

Mounted at /content/drive
Drive mounted.
Output root: /content/drive/MyDrive/gnn_qm9_final_full


In [2]:

# ── Cell 2: Install Dependencies ─────────────────────────────────────────────
#
# PyTorch Geometric and its sparse backends (torch-scatter, torch-sparse)
# must match the exact PyTorch and CUDA versions already on the runtime.
# We detect those versions dynamically so this cell works regardless of
# which Colab runtime you're on.

import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device     :', torch.cuda.get_device_name(0))

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = 'cu128' if torch.cuda.is_available() else 'cpu'

!pip install -q torch-geometric
!pip install -q pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html
!pip install -q pyyaml tqdm

print('\nInstallation complete.')

PyTorch version : 2.10.0+cu128
CUDA available  : True
CUDA device     : NVIDIA A100-SXM4-40GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 84.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 52.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 112.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 142.4 MB/s eta 0:00:00

Installation complete.


In [3]:
# ── Cell 3: Clone Project Repo ───────────────────────────────────────────────
#
# Clones your project into /content/gnn_qm9 and adds it to sys.path so
# all local imports (data.loader, models, train, evaluate) resolve correctly.
#
# If you've already cloned in a previous session and the directory exists,
# we do a git pull instead to pick up any changes you pushed.

import os, sys

PROJECT_DIR = '/content/gnn_qm9'
REPO_URL    = 'https://github.com/amanikonda123/DL-Final-Project.git'

if os.path.exists(PROJECT_DIR):
    print('Repo already exists — pulling latest changes...')
    !git -C {PROJECT_DIR} pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Working directory:', os.getcwd())
print('Project files    :', os.listdir('.'))

Cloning into '/content/gnn_qm9'...
remote: Enumerating objects: 500, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 500 (delta 4), reused 30 (delta 3), pack-reused 460 (from 1)
Receiving objects: 100% (500/500), 69.86 MiB | 18.11 MiB/s, done.
Resolving deltas: 100% (190/190), done.
Working directory: /content/gnn_qm9
Project files    : ['visualize_architectures.ipynb', 'ablation_study1_layer_depth.ipynb', '.git', 'models', 'tune_gat.ipynb', 'gatv2_best.pt', 'README.md', 'requirements.txt', 'ablation_study4_edge_features.ipynb', 'train_colab.ipynb', 'train.py', 'ablation_study3_identity_adjacency.ipynb', 'evaluation', 'DL_Final_Project_Base.ipynb', '.DS_Store', 'ablation_study.ipynb', 'training', 'train_schnet.py', 'outputs', '.gitignore', 'config', 'utils', 'ablation_study2_feature_mode.ipynb', 'data', '.vscode', 'evaluate.py']


In [4]:
# ── Cell 4: Setup — Device, Paths, Shared Data Loaders ───────────────────────
#
# We load data once and reuse the same train/val/test splits for every trial.
# This is important: if each trial loaded its own splits, random seed differences
# could mean different models see different training molecules, making comparisons
# unfair. One shared loader with a fixed seed guarantees all trials are evaluated
# on identical splits.
#
# We use the GCN config only to establish the splits and normalizer — the actual
# feature preparation (x, edge_attr) happens inside each trial's training loop
# via select_features(), so the loader itself is feature-mode agnostic.
#
# TUNING BUDGET: Set NUM_TRIALS to control how many random configs are sampled
# per model. 25 trials per model (50 total) is a practical Colab budget.
# Increase if you have a GPU runtime with more time available.

import torch
import yaml
import random
import numpy as np
from data.loader import get_dataloaders

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT  = f'{DRIVE_ROOT}/data/qm9_raw'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'
NUM_TRIALS = 1   # ← adjust based on available compute

print(f'Device      : {DEVICE}')
print(f'Num trials  : {NUM_TRIALS} per model ({NUM_TRIALS * 2} total)')

# Load base configs — used for dataset/training settings shared across all trials
with open('config/gat.yaml')   as f: GAT_BASE_CFG   = yaml.safe_load(f)
with open('config/gatv2.yaml') as f: GATV2_BASE_CFG = yaml.safe_load(f)

# Shared data loaders — same splits and normalizer for every trial
train_loader, val_loader, test_loader, normalizer = get_dataloaders(
    GAT_BASE_CFG, root=DATA_ROOT
)

# Sanity check
batch = next(iter(train_loader))
print(f'\nTrain batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')
print(f'Batch x shape : {batch.x.shape}')           # [N, 5]
print(f'Batch ea shape: {batch.edge_attr.shape}')    # [E, 4]

Device      : cuda
Num trials  : 1 per model (2 total)
[loader] Target: U0 (Internal energy at 0K, Ha)
[loader] Loading QM9 from '/content/drive/MyDrive/gnn_qm9_final_full/data/qm9_raw'...


Extracting /content/drive/MyDrive/gnn_qm9_final_full/data/qm9_raw/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


[loader] Train target mean=-11183.0596, std=1081.5560
[loader] Split sizes — train: 104664, val: 13083, test: 13084

Train batches : 818
Val   batches : 103
Test  batches : 103
Batch x shape : torch.Size([2291, 11])
Batch ea shape: torch.Size([4728, 4])


In [5]:
# ── Cell 5: Search Space and Random Sampler ───────────────────────────────────
#
# The search space is defined as a dict of hyperparameter name → list of
# candidate values. The sampler draws one value per key uniformly at random.
#
# Invalid combinations (hidden_dim not divisible by heads) are rejected and
# resampled — we keep drawing until we have exactly NUM_TRIALS valid configs.
#
# share_weights is a GATv2-only hyperparameter. For GAT trials it is simply
# ignored when building the config dict passed to train_model.
#
# We fix the dataset and training settings (epochs, scheduler, patience,
# batch_size) from the base config and only vary the model architecture
# and learning rate. This keeps the comparison focused on what matters.

import copy
import random

SEARCH_SPACE = {
    'hidden_dim'   : [256, 256],
    'num_layers'   : [6, 6],
    'heads'        : [8, 8],
    'dropout'      : [0.0],
    'lr'           : [5e-4],
    'share_weights': [False],   # GATv2 only
}

def sample_config(base_cfg: dict, model_name: str, seed: int) -> dict:
    """
    Draw one valid random configuration from SEARCH_SPACE.
    Resamples until hidden_dim % heads == 0.
    Returns a full config dict ready to pass to train_model.
    """
    random.seed(seed)
    while True:
        hidden_dim = random.choice(SEARCH_SPACE['hidden_dim'])
        heads      = random.choice(SEARCH_SPACE['heads'])
        if hidden_dim % heads != 0:
            continue   # invalid — resample

        cfg = copy.deepcopy(base_cfg)
        cfg['model']['hidden_dim'] = hidden_dim
        cfg['model']['num_layers'] = random.choice(SEARCH_SPACE['num_layers'])
        cfg['model']['heads']      = heads
        cfg['model']['dropout']    = random.choice(SEARCH_SPACE['dropout'])
        cfg['training']['lr']      = random.choice(SEARCH_SPACE['lr'])

        if model_name == 'gatv2':
            cfg['model']['share_weights'] = random.choice(SEARCH_SPACE['share_weights'])
            cfg['model']['edge_dim']      = 4   # fixed for topology mode

        return cfg


def generate_trial_configs(base_cfg: dict, model_name: str, n: int, base_seed: int = 0):
    """
    Generate n unique valid configs by using incrementing seeds.
    Returns a list of (trial_id, config) tuples.
    """
    configs = []
    seen    = set()
    seed    = base_seed

    while len(configs) < n:
        cfg = sample_config(base_cfg, model_name, seed)
        key = str(cfg['model'])   # deduplicate by model section
        if key not in seen:
            seen.add(key)
            trial_id = f'{model_name}_trial_{len(configs):03d}'
            configs.append((trial_id, cfg))
        seed += 1

    return configs


# Preview a few samples
preview_gat   = generate_trial_configs(GAT_BASE_CFG,   'gat',   n=3, base_seed=0)
preview_gatv2 = generate_trial_configs(GATV2_BASE_CFG, 'gatv2', n=3, base_seed=0)

print('Example GAT configs:')
for tid, cfg in preview_gat:
    print(f'  {tid}: model={cfg["model"]}  lr={cfg["training"]["lr"]}')

print('\nExample GATv2 configs:')
for tid, cfg in preview_gatv2:
    print(f'  {tid}: model={cfg["model"]}  lr={cfg["training"]["lr"]}')

KeyboardInterrupt: 

In [6]:
# Cell 5.5: Final Config
import copy

cfg_gat = copy.deepcopy(GAT_BASE_CFG)
cfg_gat['model']['hidden_dim']  = 256
cfg_gat['model']['num_layers']  = 6
cfg_gat['model']['heads']       = 8
cfg_gat['model']['dropout']     = 0.0
cfg_gat['training']['lr']       = 5e-4
cfg_gat['dataset']['feature_mode']  = 'full'


cfg_gatv2 = copy.deepcopy(GATV2_BASE_CFG)
cfg_gatv2['model']['hidden_dim']    = 256
cfg_gatv2['model']['num_layers']    = 6
cfg_gatv2['model']['heads']         = 8
cfg_gatv2['model']['dropout']       = 0.0
cfg_gatv2['model']['edge_dim']      = 4
cfg_gatv2['model']['share_weights'] = False
cfg_gatv2['training']['lr']         = 5e-4

print('GAT config:',   cfg_gat['model'])
print('GATv2 config:', cfg_gatv2['model'])

GAT config: {'hidden_dim': 256, 'num_layers': 6, 'dropout': 0.0, 'heads': 8}
GATv2 config: {'hidden_dim': 256, 'num_layers': 6, 'dropout': 0.0, 'heads': 8, 'edge_dim': 4, 'share_weights': False}


In [7]:
print('GAT config:',   cfg_gat['dataset'])


GAT config: {'target': 7, 'split': [0.8, 0.1, 0.1], 'seed': 42, 'feature_mode': 'full'}


In [ ]:
# # ── Cell 6: Run Random Search — GAT ──────────────────────────────────────────
# #
# # For each sampled configuration we:
# #   1. Train the model using the shared data loaders
# #   2. Log per-epoch train_loss, val_loss (MSE), val_mae to a per-trial CSV
# #   3. Save the best checkpoint (lowest val_mae) to Drive
# #   4. Record the trial summary (hyperparams + best_val_mae) in a master CSV
# #
# # Validation metrics are computed after every epoch inside train_model —
# # see training/trainer.py. The best_val_mae returned is the minimum across
# # all epochs for that trial, corresponding to the saved checkpoint.
# #
# # The master CSV (gat_tuning_results.csv) is written incrementally so that
# # if the session crashes mid-run, completed trials are not lost.
# #
# # Expected runtime: ~2–5 min per trial on a T4 GPU, so 25 trials ≈ 1–2 hours.

# import csv
# import shutil
# from train import train_model
# from tqdm import tqdm

# GAT_TRIALS     = generate_trial_configs(GAT_BASE_CFG, 'gat', n=NUM_TRIALS, base_seed=100)
# GAT_RESULTS    = []
# GAT_MASTER_CSV = f'{OUTPUT_DIR}/results/gat_tuning_results.csv'

# master_fields = [
#     'trial_id', 'hidden_dim', 'num_layers', 'heads', 'dropout', 'lr',
#     'best_val_mae', 'checkpoint',
# ]
# with open(GAT_MASTER_CSV, 'w', newline='') as f:
#     csv.DictWriter(f, fieldnames=master_fields).writeheader()

# print(f'Running {NUM_TRIALS} GAT trials...\n')

# for trial_id, cfg in tqdm(GAT_TRIALS, desc='GAT trials'):
#     mcfg = cfg['model']
#     print(f'\n── {trial_id} ──')
#     print(f'   hidden={mcfg["hidden_dim"]}  layers={mcfg["num_layers"]}  '
#           f'heads={mcfg["heads"]}  dropout={mcfg["dropout"]}  lr={cfg["training"]["lr"]}')

#     result = train_model(
#         cfg,
#         device=str(DEVICE),
#         output_dir=f'{OUTPUT_DIR}/checkpoints/tuning/gat',
#         data_root=DATA_ROOT,
#         model_name='gat',
#         loaders = (train_loader, val_loader, normalizer),
#     )

#     # Copy checkpoint to trial-specific name before next trial overwrites it
#     named_ckpt = f'{OUTPUT_DIR}/checkpoints/tuning/gat/{trial_id}_best.pt'
#     shutil.copy(result['checkpoint'], named_ckpt)
#     result['checkpoint'] = named_ckpt

#     # Save per-trial history CSV
#     log_path = f'{OUTPUT_DIR}/logs/tuning/gat/{trial_id}_history.csv'
#     with open(log_path, 'w', newline='') as f:
#         writer = csv.DictWriter(f, fieldnames=['epoch', 'train_loss', 'val_loss', 'val_mae', 'lr'])
#         writer.writeheader()
#         writer.writerows(result['history'])

#     row = {
#         'trial_id'    : trial_id,
#         'hidden_dim'  : mcfg['hidden_dim'],
#         'num_layers'  : mcfg['num_layers'],
#         'heads'       : mcfg['heads'],
#         'dropout'     : mcfg['dropout'],
#         'lr'          : cfg['training']['lr'],
#         'best_val_mae': result['best_val_mae'],
#         'checkpoint'  : result['checkpoint'],
#     }
#     GAT_RESULTS.append(row)

#     with open(GAT_MASTER_CSV, 'a', newline='') as f:
#         csv.DictWriter(f, fieldnames=master_fields).writerow(row)

#     print(f'   best_val_mae = {result["best_val_mae"]:.6f}')

# GAT_RESULTS.sort(key=lambda r: r['best_val_mae'])
# print('\n── GAT leaderboard (top 5) ──')
# for r in GAT_RESULTS[:5]:
#     print(f'  {r["trial_id"]}  MAE={r["best_val_mae"]:.6f}  '
#           f'hidden={r["hidden_dim"]}  layers={r["num_layers"]}  '
#           f'heads={r["heads"]}  dropout={r["dropout"]}  lr={r["lr"]}')

In [8]:
# ── Cell 6.5: Train GAT — Single Config ────────────────────────────────────────
import csv
import shutil
from train import train_model

TRIAL_ID = 'gat_final'

print(f'Training GAT with config: {cfg_gat["model"]}\n')

result = train_model(
    cfg_gat,
    device=str(DEVICE),
    output_dir=f'{OUTPUT_DIR}/checkpoints',
    data_root=DATA_ROOT,
    model_name='gat',
    loaders=(train_loader, val_loader, normalizer),
)

# Save checkpoint
named_ckpt = f'{OUTPUT_DIR}/checkpoints/{TRIAL_ID}_best.pt'
shutil.copy(result['checkpoint'], named_ckpt)

# Save training history
log_path = f'{OUTPUT_DIR}/logs/{TRIAL_ID}_history.csv'
with open(log_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['epoch', 'train_loss', 'val_loss', 'val_mae', 'lr'])
    writer.writeheader()
    writer.writerows(result['history'])

print(f'Best val MAE: {result["best_val_mae"]:.6f}')
print(f'Checkpoint:  {named_ckpt}')
print(f'History:     {log_path}')

Training GAT with config: {'hidden_dim': 256, 'num_layers': 6, 'dropout': 0.0, 'heads': 8}

[train] Model: gat  |  params: 404,737


Training gat: 100%|██████████| 200/200 [1:06:58<00:00, 20.09s/it, lr=3.1e-08, train_loss=0.0032, val_mae=47.7023]

[train] Best val MAE: 47.156254  — checkpoint saved to /content/drive/MyDrive/gnn_qm9_final_full/outputs/checkpoints/checkpoints/gat_best.pt
Best val MAE: 47.156254
Checkpoint:  /content/drive/MyDrive/gnn_qm9_final_full/outputs/checkpoints/gat_final_best.pt
History:     /content/drive/MyDrive/gnn_qm9_final_full/outputs/logs/gat_final_history.csv


In [ ]:
# # ── Cell 7: Run Random Search — GATv2 ────────────────────────────────────────
# #
# # Identical structure to Cell 6 but for GATv2. The only additional
# # hyperparameter is share_weights — whether W_l and W_r are tied.
# #
# # GATv2 has more parameters per trial than GAT (split W_l/W_r + W_e for
# # edge features) so each trial may be slightly slower. The same NUM_TRIALS
# # budget applies.
# #
# # Results are written to gatv2_tuning_results.csv with an extra
# # share_weights column.

# import csv
# import shutil
# from train import train_model
# from tqdm import tqdm

# GATV2_TRIALS     = generate_trial_configs(GATV2_BASE_CFG, 'gatv2', n=NUM_TRIALS, base_seed=200)
# GATV2_RESULTS    = []
# GATV2_MASTER_CSV = f'{OUTPUT_DIR}/results/gatv2_tuning_results.csv'

# master_fields_v2 = [
#     'trial_id', 'hidden_dim', 'num_layers', 'heads', 'dropout', 'lr',
#     'share_weights', 'best_val_mae', 'checkpoint',
# ]
# with open(GATV2_MASTER_CSV, 'w', newline='') as f:
#     csv.DictWriter(f, fieldnames=master_fields_v2).writeheader()

# print(f'Running {NUM_TRIALS} GATv2 trials...\n')

# for trial_id, cfg in tqdm(GATV2_TRIALS, desc='GATv2 trials'):
#     mcfg = cfg['model']
#     print(f'\n── {trial_id} ──')
#     print(f'   hidden={mcfg["hidden_dim"]}  layers={mcfg["num_layers"]}  '
#           f'heads={mcfg["heads"]}  dropout={mcfg["dropout"]}  '
#           f'lr={cfg["training"]["lr"]}  share_weights={mcfg["share_weights"]}')

#     result = train_model(
#         cfg,
#         device=str(DEVICE),
#         output_dir=f'{OUTPUT_DIR}/checkpoints/tuning/gatv2',
#         data_root=DATA_ROOT,
#         model_name='gatv2',
#         loaders = (train_loader, val_loader, normalizer),
#     )

#     # Copy checkpoint to trial-specific name before next trial overwrites it
#     named_ckpt = f'{OUTPUT_DIR}/checkpoints/tuning/gatv2/{trial_id}_best.pt'
#     shutil.copy(result['checkpoint'], named_ckpt)
#     result['checkpoint'] = named_ckpt

#     # Save per-trial history CSV
#     log_path = f'{OUTPUT_DIR}/logs/tuning/gatv2/{trial_id}_history.csv'
#     with open(log_path, 'w', newline='') as f:
#         writer = csv.DictWriter(f, fieldnames=['epoch', 'train_loss', 'val_loss', 'val_mae', 'lr'])
#         writer.writeheader()
#         writer.writerows(result['history'])

#     row = {
#         'trial_id'     : trial_id,
#         'hidden_dim'   : mcfg['hidden_dim'],
#         'num_layers'   : mcfg['num_layers'],
#         'heads'        : mcfg['heads'],
#         'dropout'      : mcfg['dropout'],
#         'lr'           : cfg['training']['lr'],
#         'share_weights': mcfg['share_weights'],
#         'best_val_mae' : result['best_val_mae'],
#         'checkpoint'   : result['checkpoint'],
#     }
#     GATV2_RESULTS.append(row)

#     with open(GATV2_MASTER_CSV, 'a', newline='') as f:
#         csv.DictWriter(f, fieldnames=master_fields_v2).writerow(row)

#     print(f'   best_val_mae = {result["best_val_mae"]:.6f}')

# GATV2_RESULTS.sort(key=lambda r: r['best_val_mae'])
# print('\n── GATv2 leaderboard (top 5) ──')
# for r in GATV2_RESULTS[:5]:
#     print(f'  {r["trial_id"]}  MAE={r["best_val_mae"]:.6f}  '
#           f'hidden={r["hidden_dim"]}  layers={r["num_layers"]}  '
#           f'heads={r["heads"]}  dropout={r["dropout"]}  '
#           f'lr={r["lr"]}  share={r["share_weights"]}')

In [ ]:
# # ── Cell 8: Hyperparameter Sensitivity Plots ──────────────────────────────────
# #
# # For each tunable hyperparameter we plot its value on the x-axis and the
# # best_val_mae of every trial that used that value on the y-axis.
# #
# # Each subplot shows both GAT (blue) and GATv2 (orange) as grouped boxplots.
# # A boxplot is more informative than a mean line here because with 25 trials
# # and 3 values per hyperparameter, each box covers roughly 8 trials — enough
# # to show the distribution of outcomes for that setting, not just the average.
# #
# # Reading these plots: a lower median box means that value tended to produce
# # better models. A wide box means the outcome was sensitive to other hyperparams
# # too — that setting alone wasn't sufficient for good performance.

# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker

# df_gat   = pd.DataFrame(GAT_RESULTS)
# df_gatv2 = pd.DataFrame(GATV2_RESULTS)

# SHARED_HPARAMS = ['hidden_dim', 'num_layers', 'heads', 'dropout', 'lr']

# fig, axes = plt.subplots(1, len(SHARED_HPARAMS), figsize=(22, 5))
# fig.suptitle('Hyperparameter Sensitivity — GAT vs GATv2\n(lower val MAE is better)',
#              fontsize=13, y=1.02)

# for ax, hp in zip(axes, SHARED_HPARAMS):
#     vals = sorted(df_gat[hp].unique())
#     x    = range(len(vals))

#     gat_groups   = [df_gat[df_gat[hp] == v]['best_val_mae'].values   for v in vals]
#     gatv2_groups = [df_gatv2[df_gatv2[hp] == v]['best_val_mae'].values for v in vals]

#     width  = 0.35
#     pos_gat   = [i - width/2 for i in x]
#     pos_gatv2 = [i + width/2 for i in x]

#     bp1 = ax.boxplot(gat_groups,   positions=pos_gat,   widths=0.3,
#                      patch_artist=True, boxprops=dict(facecolor='#4C72B0', alpha=0.7),
#                      medianprops=dict(color='white', linewidth=2),
#                      whiskerprops=dict(color='#4C72B0'),
#                      capprops=dict(color='#4C72B0'),
#                      flierprops=dict(marker='o', color='#4C72B0', markersize=3))
#     bp2 = ax.boxplot(gatv2_groups, positions=pos_gatv2, widths=0.3,
#                      patch_artist=True, boxprops=dict(facecolor='#DD8452', alpha=0.7),
#                      medianprops=dict(color='white', linewidth=2),
#                      whiskerprops=dict(color='#DD8452'),
#                      capprops=dict(color='#DD8452'),
#                      flierprops=dict(marker='o', color='#DD8452', markersize=3))

#     ax.set_xticks(list(x))
#     ax.set_xticklabels([str(v) for v in vals], fontsize=9)
#     ax.set_xlabel(hp, fontsize=10)
#     ax.set_ylabel('Val MAE' if hp == SHARED_HPARAMS[0] else '')
#     ax.set_title(hp, fontsize=10)
#     ax.grid(axis='y', alpha=0.3)
#     ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.4f'))

# # Shared legend
# from matplotlib.patches import Patch
# legend_elements = [
#     Patch(facecolor='#4C72B0', alpha=0.7, label='GAT'),
#     Patch(facecolor='#DD8452', alpha=0.7, label='GATv2'),
# ]
# fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.0, 1.0), fontsize=10)

# plt.tight_layout()
# save_path = f'{OUTPUT_DIR}/plots/tuning/sensitivity.png'
# plt.savefig(save_path, dpi=150, bbox_inches='tight')
# plt.show()
# print(f'Saved to {save_path}')

In [ ]:
# # ── Cell 9: Pairwise Heatmap — hidden_dim × num_layers ───────────────────────
# #
# # A heatmap shows whether hidden_dim and num_layers interact — i.e., whether
# # a large hidden_dim only helps if you also have more layers, or vice versa.
# # Each cell shows the median val MAE across all trials with that combination
# # of the two hyperparameters (other hyperparams vary freely across those trials).
# #
# # If the heatmap is mostly flat (all cells similar colour), the two hyperparams
# # don't interact strongly. If there's a bright corner, that's the sweet spot.
# #
# # We show GAT (left) and GATv2 (right) side by side. Darker = better (lower MAE).
# # Cells with no trials are shown in grey.

# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors

# def make_heatmap(ax, df, title):
#     rows = sorted(df['num_layers'].unique())
#     cols = sorted(df['hidden_dim'].unique())
#     grid = np.full((len(rows), len(cols)), np.nan)

#     for i, nl in enumerate(rows):
#         for j, hd in enumerate(cols):
#             subset = df[(df['num_layers'] == nl) & (df['hidden_dim'] == hd)]['best_val_mae']
#             if len(subset) > 0:
#                 grid[i, j] = subset.median()

#     # Mask NaN cells
#     masked = np.ma.masked_invalid(grid)
#     cmap = plt.cm.viridis_r.copy()
#     cmap.set_bad(color='lightgrey')

#     im = ax.imshow(masked, cmap=cmap, aspect='auto')
#     plt.colorbar(im, ax=ax, label='Median val MAE')

#     ax.set_xticks(range(len(cols)))
#     ax.set_yticks(range(len(rows)))
#     ax.set_xticklabels([str(c) for c in cols])
#     ax.set_yticklabels([str(r) for r in rows])
#     ax.set_xlabel('hidden_dim')
#     ax.set_ylabel('num_layers')
#     ax.set_title(title)

#     # Annotate cells with median value
#     for i in range(len(rows)):
#         for j in range(len(cols)):
#             if not np.isnan(grid[i, j]):
#                 ax.text(j, i, f'{grid[i,j]:.4f}', ha='center', va='center',
#                         fontsize=8, color='white')

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# fig.suptitle('Pairwise Heatmap: hidden_dim × num_layers\n(median val MAE — darker = better)',
#              fontsize=12)

# make_heatmap(axes[0], df_gat,   'GAT')
# make_heatmap(axes[1], df_gatv2, 'GATv2')

# plt.tight_layout()
# save_path = f'{OUTPUT_DIR}/plots/tuning/heatmap_hidden_layers.png'
# plt.savefig(save_path, dpi=150, bbox_inches='tight')
# plt.show()
# print(f'Saved to {save_path}')

In [ ]:
# # ── Cell 10: GAT vs GATv2 Head-to-Head Scatter ────────────────────────────────
# #
# # For each hyperparameter combination that appears in both GAT and GATv2 trials,
# # we plot a point with GAT val MAE on the x-axis and GATv2 val MAE on the y-axis.
# #
# # Points BELOW the diagonal (y < x) mean GATv2 won for that config.
# # Points ABOVE the diagonal mean GAT won.
# #
# # If most points are below the diagonal, GATv2's dynamic attention and edge
# # features are genuinely helping on QM9. If they're scattered around the
# # diagonal, the architectural difference isn't consistently translating to
# # better performance — which would be worth investigating.
# #
# # Shared hyperparameters (hidden_dim, num_layers, heads, dropout, lr) are
# # used to match trials. Because GAT and GATv2 were sampled independently
# # with different seeds, exact matches may be sparse — we match on the
# # closest GAT config for each GATv2 config using pandas merge.

# import matplotlib.pyplot as plt

# MATCH_COLS = ['hidden_dim', 'num_layers', 'heads', 'dropout', 'lr']

# merged = pd.merge(
#     df_gat[MATCH_COLS   + ['best_val_mae']].rename(columns={'best_val_mae': 'gat_mae'}),
#     df_gatv2[MATCH_COLS + ['best_val_mae']].rename(columns={'best_val_mae': 'gatv2_mae'}),
#     on=MATCH_COLS,
#     how='inner',
# )

# print(f'Matched {len(merged)} shared configurations')

# fig, ax = plt.subplots(figsize=(7, 7))

# ax.scatter(merged['gat_mae'], merged['gatv2_mae'],
#            alpha=0.7, s=60, color='steelblue', edgecolors='white', linewidths=0.5)

# # Diagonal — equal performance line
# lim_min = min(merged['gat_mae'].min(), merged['gatv2_mae'].min()) * 0.98
# lim_max = max(merged['gat_mae'].max(), merged['gatv2_mae'].max()) * 1.02
# ax.plot([lim_min, lim_max], [lim_min, lim_max],
#         'k--', linewidth=1, alpha=0.5, label='Equal performance')

# # Shade regions
# ax.fill_between([lim_min, lim_max], [lim_min, lim_min], [lim_min, lim_max],
#                 alpha=0.05, color='orange', label='GATv2 better')
# ax.fill_between([lim_min, lim_max], [lim_min, lim_max], [lim_max, lim_max],
#                 alpha=0.05, color='blue',   label='GAT better')

# n_gatv2_wins = (merged['gatv2_mae'] < merged['gat_mae']).sum()
# n_gat_wins   = (merged['gat_mae'] < merged['gatv2_mae']).sum()

# ax.set_xlabel('GAT val MAE',   fontsize=12)
# ax.set_ylabel('GATv2 val MAE', fontsize=12)
# ax.set_title(f'GAT vs GATv2 — matched configs\n'
#              f'GATv2 wins: {n_gatv2_wins}/{len(merged)}  '
#              f'GAT wins: {n_gat_wins}/{len(merged)}', fontsize=11)
# ax.legend(fontsize=9)
# ax.grid(alpha=0.3)
# ax.set_xlim(lim_min, lim_max)
# ax.set_ylim(lim_min, lim_max)

# plt.tight_layout()
# save_path = f'{OUTPUT_DIR}/plots/tuning/head_to_head_scatter.png'
# plt.savefig(save_path, dpi=150, bbox_inches='tight')
# plt.show()
# print(f'Saved to {save_path}')

In [ ]:
# # ── Cell 11: Parameter Count vs Val MAE ───────────────────────────────────────
# #
# # We compute the total trainable parameter count for each trial configuration
# # and plot it against the achieved val MAE. This answers a key question:
# # are GATv2's extra parameters (from split W_l/W_r and W_e for edge features)
# # buying proportional performance improvement, or is it just more parameters
# # without meaningful gain?
# #
# # We reconstruct the model from the saved config for each trial and call
# # sum(p.numel() for p in model.parameters()) — we do not load the checkpoint
# # weights since parameter count depends only on architecture, not weights.
# #
# # An ideal result: GATv2 points cluster toward the lower-right (more params,
# # lower MAE), showing the extra capacity is being used. If GATv2 points are
# # just shifted right with no downward shift, the extra parameters aren't helping.

# import matplotlib.pyplot as plt
# from models import build_model

# def get_param_count(model_name: str, cfg: dict) -> int:
#     """Build model from config and return trainable parameter count."""
#     from data.features import get_feature_dims
#     feature_dims = get_feature_dims(cfg['dataset'].get('feature_mode', 'topology'))
#     model = build_model(model_name, cfg, feature_dims=feature_dims)
#     return sum(p.numel() for p in model.parameters() if p.requires_grad)

# # Reattach configs to results so we can rebuild models
# gat_with_cfg   = list(zip([r for r in GAT_RESULTS],   [cfg for _, cfg in GAT_TRIALS]))
# gatv2_with_cfg = list(zip([r for r in GATV2_RESULTS], [cfg for _, cfg in GATV2_TRIALS]))

# # Sort by trial_id to align with results list order
# gat_param_counts   = [get_param_count('gat',   cfg) for _, cfg in GAT_TRIALS]
# gatv2_param_counts = [get_param_count('gatv2', cfg) for _, cfg in GATV2_TRIALS]

# gat_maes   = [r['best_val_mae'] for r in GAT_RESULTS]
# gatv2_maes = [r['best_val_mae'] for r in GATV2_RESULTS]

# fig, ax = plt.subplots(figsize=(9, 6))

# ax.scatter(gat_param_counts,   gat_maes,   alpha=0.7, s=60,
#            color='#4C72B0', label='GAT',   edgecolors='white', linewidths=0.5)
# ax.scatter(gatv2_param_counts, gatv2_maes, alpha=0.7, s=60,
#            color='#DD8452', label='GATv2', edgecolors='white', linewidths=0.5)

# ax.set_xlabel('Trainable parameter count', fontsize=12)
# ax.set_ylabel('Best val MAE',              fontsize=12)
# ax.set_title('Parameter count vs validation MAE\n'
#              'Lower-right is better (fewer params, lower error)',
#              fontsize=11)
# ax.legend(fontsize=10)
# ax.grid(alpha=0.3)
# ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

# plt.tight_layout()
# save_path = f'{OUTPUT_DIR}/plots/tuning/params_vs_mae.png'
# plt.savefig(save_path, dpi=150, bbox_inches='tight')
# plt.show()
# print(f'Saved to {save_path}')

In [ ]:
# # ── Cell 12: Test Set Evaluation — Best Config per Model ─────────────────────
# #
# # We evaluate on the test set exactly once per model, using the best
# # checkpoint identified from the validation leaderboard.
# #
# # IMPORTANT: the test set is only touched here, after all hyperparameter
# # selection is complete. Using test metrics to guide any earlier decision
# # (including which cell to rerun) would constitute test set leakage and
# # would make the reported test MAE an optimistic overestimate of true
# # generalisation performance.
# #
# # We report both MAE and RMSE. MAE is the primary metric for QM9 comparisons
# # in the literature. RMSE is more sensitive to outlier predictions and gives
# # a complementary view of worst-case error.
# #
# # Results are saved to results/best_model_test_results.csv.

# from evaluate import evaluate_model
# import json

# # Best trial = first entry after sorting by val MAE in Cells 6 and 7
# best_gat_row   = GAT_RESULTS[0]
# best_gatv2_row = GATV2_RESULTS[0]

# print('Best GAT config  :', {k: best_gat_row[k]   for k in ['hidden_dim','num_layers','heads','dropout','lr']})
# print('Best GATv2 config:', {k: best_gatv2_row[k] for k in ['hidden_dim','num_layers','heads','dropout','lr','share_weights']})
# print()

# # Rebuild the best configs
# best_gat_cfg   = [cfg for tid, cfg in GAT_TRIALS   if tid == best_gat_row['trial_id']][0]
# best_gatv2_cfg = [cfg for tid, cfg in GATV2_TRIALS if tid == best_gatv2_row['trial_id']][0]

# test_results = {}

# for model_name, cfg, row in [
#     ('gat',   best_gat_cfg,   best_gat_row),
#     ('gatv2', best_gatv2_cfg, best_gatv2_row),
# ]:
#     print(f'── Evaluating {model_name.upper()} on test set ──')
#     result = evaluate_model(
#         cfg,
#         checkpoint_path=row['checkpoint'],
#         device=str(DEVICE),
#         output_dir=OUTPUT_DIR,
#         data_root=DATA_ROOT,
#         model_name=model_name,
#         loaders=(test_loader, normalizer), # pass shared loaders
#     )
#     test_results[model_name] = result
#     print(f'   Test MAE  : {result["test_mae"]:.6f}')
#     print(f'   Test RMSE : {result["test_rmse"]:.6f}')
#     print()

# # Save summary
# summary_path = f'{OUTPUT_DIR}/results/best_model_test_results.json'
# with open(summary_path, 'w') as f:
#     json.dump({
#         'gat': {
#             'config'    : best_gat_row,
#             'test_mae'  : test_results['gat']['test_mae'],
#             'test_rmse' : test_results['gat']['test_rmse'],
#         },
#         'gatv2': {
#             'config'    : best_gatv2_row,
#             'test_mae'  : test_results['gatv2']['test_mae'],
#             'test_rmse' : test_results['gatv2']['test_rmse'],
#         },
#     }, f, indent=2)

# print(f'Results saved to {summary_path}')

# print('\n── Final Test Results ──')
# print(f'{"Model":<10} {"Test MAE":>12} {"Test RMSE":>12}')
# print('-' * 36)
# for name in ['gat', 'gatv2']:
#     r = test_results[name]
#     print(f'{name.upper():<10} {r["test_mae"]:>12.6f} {r["test_rmse"]:>12.6f}')

In [10]:
# ── Cell 12.5: Test Set Evaluation — GAT ───────────────────────────────────────
from evaluate import evaluate_model
import json

print(f'Best GAT config: {cfg_gat["model"]}\n')

print('── Evaluating GAT on test set ──')
result = evaluate_model(
    cfg_gat,
    checkpoint_path=named_ckpt,
    device=str(DEVICE),
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    model_name='gat',
    loaders=(test_loader, normalizer),
)
print(f'   Test MAE  : {result["test_mae"]:.6f}')
print(f'   Test RMSE : {result["test_rmse"]:.6f}')

# Save summary
summary_path = f'{OUTPUT_DIR}/results/best_model_test_results_GAT.json'
with open(summary_path, 'w') as f:
    json.dump({
        'gat': {
            'config'   : cfg_gat['model'],
            'test_mae' : result['test_mae'],
            'test_rmse': result['test_rmse'],
        },
    }, f, indent=2)

print(f'\nResults saved to {summary_path}')

print('\n── Final Test Results ──')
print(f'{"Model":<10} {"Test MAE":>12} {"Test RMSE":>12}')
print('-' * 36)
print(f'{"GAT":<10} {result["test_mae"]:>12.6f} {result["test_rmse"]:>12.6f}')

Best GAT config: {'hidden_dim': 256, 'num_layers': 6, 'dropout': 0.0, 'heads': 8}

── Evaluating GAT on test set ──
[eval] Test MAE  = 45.974068
[eval] Test RMSE = 175.412433
[eval] Samples   = 13084
[eval] Predictions saved to /content/drive/MyDrive/gnn_qm9_final_full/outputs/results/gat_predictions.csv
   Test MAE  : 45.974068
   Test RMSE : 175.412433

Results saved to /content/drive/MyDrive/gnn_qm9_final_full/outputs/results/best_model_test_results_GAT.json

── Final Test Results ──
Model          Test MAE    Test RMSE
------------------------------------
GAT           45.974068   175.412433


In [12]:
print(result['predictions_path'])

/content/drive/MyDrive/gnn_qm9_final_full/outputs/results/gat_predictions.csv


In [14]:
# Cell: Test Set Evaluation — GATv2 ──────────────────────────────────────────
from evaluate import evaluate_model

cfg_gatv2 = copy.deepcopy(GATV2_BASE_CFG)
cfg_gatv2['model']['hidden_dim']    = 256
cfg_gatv2['model']['num_layers']    = 6
cfg_gatv2['model']['heads']         = 8
cfg_gatv2['model']['dropout']       = 0.0
cfg_gatv2['model']['edge_dim']      = 4
cfg_gatv2['model']['share_weights'] = False
cfg_gatv2['training']['lr']         = 5e-4

GATV2_CKPT = f'{DRIVE_ROOT}/outputs/checkpoints/gatv2_best.pt'  # ← update this

print('── Evaluating GATv2 on test set ──')
result_gatv2 = evaluate_model(
    cfg_gatv2,
    checkpoint_path=GATV2_CKPT,
    device=str(DEVICE),
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    model_name='gatv2',
    loaders=(test_loader, normalizer),
)
print(f'   Test MAE  : {result_gatv2["test_mae"]:.6f}')
print(f'   Test RMSE : {result_gatv2["test_rmse"]:.6f}')
print(f'   Predictions: {result_gatv2["predictions_path"]}')

── Evaluating GATv2 on test set ──
[eval] Test MAE  = 24.471449
[eval] Test RMSE = 362.245665
[eval] Samples   = 13084
[eval] Predictions saved to /content/drive/MyDrive/gnn_qm9_final_full/outputs/results/gatv2_predictions.csv
   Test MAE  : 24.471449
   Test RMSE : 362.245665
   Predictions: /content/drive/MyDrive/gnn_qm9_final_full/outputs/results/gatv2_predictions.csv


In [ ]:
# # ── Cell 13: Training Curves — Best Config per Model ─────────────────────────
# #
# # Two plots side by side for the best GAT and best GATv2 configurations:
# #
# # Left plot — val_loss (MSE, normalised) and val_mae (denormalised) on the
# # same figure using dual y-axes. MSE and MAE tell different stories:
# # MSE is more sensitive to large errors (outlier molecules), MAE gives a
# # cleaner picture of typical prediction error. Showing both lets you spot
# # if the model is overfitting on a few difficult molecules.
# #
# # Right plot — train_loss vs val_loss for both models on the same axes.
# # The gap between train and val loss reveals overfitting. A train loss that
# # keeps dropping while val loss plateaus or rises is the classic signature.
# #
# # The vertical dashed lines show where early stopping fired for each model.
# # If early stopping fires very early, consider reducing patience or increasing
# # the learning rate — the model may be stuck in a flat region early on.

# import pandas as pd
# import matplotlib.pyplot as plt

# def load_history(trial_id: str, model_name: str) -> pd.DataFrame:
#     """Load the per-epoch training log CSV for a given trial."""
#     path = f'{OUTPUT_DIR}/logs/tuning/{model_name}/{trial_id}_history.csv'
#     return pd.read_csv(path)

# h_gat   = load_history(best_gat_row['trial_id'],   'gat')
# h_gatv2 = load_history(best_gatv2_row['trial_id'], 'gatv2')

# fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# fig.suptitle('Best Configuration Training Curves', fontsize=13)

# # ── Left: dual y-axis MAE + MSE for each model separately ──
# ax_left  = axes[0]
# ax_right_twin = ax_left.twinx()

# l1, = ax_left.plot(h_gat['epoch'],   h_gat['val_mae'],
#                    color='#4C72B0', linewidth=2, label='GAT val MAE')
# l2, = ax_left.plot(h_gatv2['epoch'], h_gatv2['val_mae'],
#                    color='#DD8452', linewidth=2, label='GATv2 val MAE')
# l3, = ax_right_twin.plot(h_gat['epoch'],   h_gat['val_loss'],
#                          color='#4C72B0', linewidth=1.5, linestyle='--',
#                          alpha=0.6, label='GAT val MSE')
# l4, = ax_right_twin.plot(h_gatv2['epoch'], h_gatv2['val_loss'],
#                          color='#DD8452', linewidth=1.5, linestyle='--',
#                          alpha=0.6, label='GATv2 val MSE')

# ax_left.set_xlabel('Epoch')
# ax_left.set_ylabel('Val MAE (denormalised)', color='black')
# ax_right_twin.set_ylabel('Val MSE (normalised)', color='grey')
# ax_left.set_title('Val MAE (solid) and Val MSE (dashed)')
# ax_left.grid(alpha=0.3)
# ax_left.legend(handles=[l1, l2, l3, l4], fontsize=8, loc='upper right')

# # ── Right: train vs val loss for both models ──
# ax2 = axes[1]

# ax2.plot(h_gat['epoch'],   h_gat['train_loss'],   color='#4C72B0',
#          linewidth=2,   label='GAT train loss')
# ax2.plot(h_gat['epoch'],   h_gat['val_loss'],     color='#4C72B0',
#          linewidth=2,   linestyle='--', alpha=0.7, label='GAT val loss')
# ax2.plot(h_gatv2['epoch'], h_gatv2['train_loss'], color='#DD8452',
#          linewidth=2,   label='GATv2 train loss')
# ax2.plot(h_gatv2['epoch'], h_gatv2['val_loss'],   color='#DD8452',
#          linewidth=2,   linestyle='--', alpha=0.7, label='GATv2 val loss')

# # Early stopping lines
# ax2.axvline(x=len(h_gat),   color='#4C72B0', linestyle=':',
#             alpha=0.5, label=f'GAT stopped (epoch {len(h_gat)})')
# ax2.axvline(x=len(h_gatv2), color='#DD8452', linestyle=':',
#             alpha=0.5, label=f'GATv2 stopped (epoch {len(h_gatv2)})')

# ax2.set_xlabel('Epoch')
# ax2.set_ylabel('MSE Loss (normalised)')
# ax2.set_title('Train vs Val Loss — overfitting diagnostic')
# ax2.legend(fontsize=8)
# ax2.grid(alpha=0.3)

# plt.tight_layout()
# save_path = f'{OUTPUT_DIR}/plots/tuning/best_config_curves.png'
# plt.savefig(save_path, dpi=150, bbox_inches='tight')
# plt.show()
# print(f'Saved to {save_path}')

# print('\n── Tuning complete ──')
# print(f'All results in : {OUTPUT_DIR}/results/')
# print(f'All plots in   : {OUTPUT_DIR}/plots/tuning/')